In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "retail_operations_dev")
catalog = dbutils.widgets.get("catalog").strip()

order_items = spark.table(
    f"{catalog}.silver.order_items_current"
)

payments = spark.table(
    f"{catalog}.silver.payments_current"
).filter(F.col("status") == "captured")

enriched_items = spark.table(
    f"{catalog}.gold.order_items_customer_enriched"
)

customer_summary = spark.table(
    f"{catalog}.gold.customer_order_summary"
)

reconciliation = spark.table(
    f"{catalog}.gold.order_payment_reconciliation"
)

In [ ]:
def assert_unique(df, key_columns, table_label):
    duplicate_keys = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
    )

    assert duplicate_keys.limit(1).count() == 0, (
        f"Duplicate keys found in {table_label}: {key_columns}"
    )


assert_unique(
    enriched_items,
    ["order_id", "line_id"],
    "customer-enriched order items"
)

assert_unique(
    customer_summary,
    ["customer_id", "currency"],
    "customer order summary"
)

assert_unique(
    reconciliation,
    ["order_id", "currency"],
    "order-payment reconciliation"
)

print("Published-table uniqueness checks passed.")

In [ ]:
gold_order_fields = enriched_items.select(
    *order_items.columns
)

assert (
    order_items.exceptAll(gold_order_fields)
    .limit(1).count() == 0
), "Gold is missing or has changed some Silver order lines."

assert (
    gold_order_fields.exceptAll(order_items)
    .limit(1).count() == 0
), "Gold contains extra or different order lines."

display(
    enriched_items.groupBy(
        "product_missing",
        "customer_missing"
    ).count()
)

print("Order-line preservation checks passed.")

In [ ]:
expected_order_totals = (
    order_items
    .filter(F.col("order_status") != "cancelled")
    .groupBy("currency")
    .agg(
        F.count("*").alias("order_line_count"),
        F.sum("quantity").alias("units_ordered"),
        F.sum("line_amount").alias("ordered_value")
    )
)

published_order_totals = (
    customer_summary
    .groupBy("currency")
    .agg(
        F.sum("order_line_count").alias("order_line_count"),
        F.sum("units_ordered").alias("units_ordered"),
        F.sum("ordered_value").alias("ordered_value")
    )
)

assert (
    expected_order_totals.exceptAll(published_order_totals)
    .limit(1).count() == 0
), "Customer summary is missing or has incorrect order totals."

assert (
    published_order_totals.exceptAll(expected_order_totals)
    .limit(1).count() == 0
), "Customer summary contains unexpected order totals."

print("Customer-summary total checks passed.")

In [ ]:
expected_payment_totals = (
    payments
    .groupBy("currency")
    .agg(
        F.count("*").alias("payment_count"),
        F.sum("amount").alias("paid_total")
    )
)

published_payment_totals = (
    reconciliation
    .filter(F.col("payment_count") > 0)
    .groupBy("currency")
    .agg(
        F.sum("payment_count").alias("payment_count"),
        F.sum("paid_total").alias("paid_total")
    )
)

assert (
    expected_payment_totals.exceptAll(published_payment_totals)
    .limit(1).count() == 0
), "Reconciliation is missing or has incorrect payment totals."

assert (
    published_payment_totals.exceptAll(expected_payment_totals)
    .limit(1).count() == 0
), "Reconciliation contains unexpected payment totals."

In [ ]:
inventory_silver = spark.table(
    f"{catalog}.silver.inventory_current"
)

inventory_gold = spark.table(
    f"{catalog}.gold.inventory_availability"
)

# Each warehouse/product pair must appear once.
assert_unique(
    inventory_silver,
    ["warehouse_id", "product_id"],
    "Silver inventory"
)

assert_unique(
    inventory_gold,
    ["warehouse_id", "product_id"],
    "Gold inventory availability"
)

# Compare the original stock fields in both tables.
gold_stock_fields = inventory_gold.select(
    *inventory_silver.columns
)

assert (
    inventory_silver.exceptAll(gold_stock_fields)
    .limit(1).count() == 0
), "Gold inventory is missing or has changed Silver stock records."

assert (
    gold_stock_fields.exceptAll(inventory_silver)
    .limit(1).count() == 0
), "Gold inventory contains extra or different stock records."

# Check quantity consistency independently.
invalid_quantities = inventory_gold.filter(
    F.col("on_hand_quantity").isNull()
    | F.col("reserved_quantity").isNull()
    | F.col("available_quantity").isNull()
    | (F.col("on_hand_quantity") < 0)
    | (F.col("reserved_quantity") < 0)
    | (F.col("reserved_quantity") > F.col("on_hand_quantity"))
    | (
        F.col("available_quantity")
        != (
            F.col("on_hand_quantity")
            - F.col("reserved_quantity")
        )
    )
)

assert invalid_quantities.limit(1).count() == 0, (
    "Gold inventory contains missing or inconsistent quantities."
)

display(
    inventory_gold.groupBy(
        "product_missing",
        "stock_status"
    ).count()
)

print("Inventory publication checks passed.")

In [ ]:
published_returns = spark.table(
    f"{catalog}.silver.returns_current"
)

published_return_details = spark.table(
    f"{catalog}.gold.return_details"
)

published_order_lines = (
    spark.table(f"{catalog}.silver.order_items_current")
    .select(
        "order_id",
        "line_id",
        F.col("quantity").alias("ordered_quantity")
    )
)

# One current record per return.
duplicate_returns = (
    published_returns
    .groupBy("return_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_returns.limit(1).count() > 0:
    raise ValueError("Duplicate return IDs in Silver.")

# Every return must reference an existing order line.
missing_return_lines = published_returns.join(
    published_order_lines,
    on=["order_id", "line_id"],
    how="left_anti"
)

if missing_return_lines.limit(1).count() > 0:
    display(missing_return_lines)
    raise ValueError("Published returns reference missing order lines.")

# Recheck the combined quantity against current order quantities.
published_return_totals = (
    published_returns
    .filter(F.col("status") != "rejected")
    .groupBy("order_id", "line_id")
    .agg(
        F.sum(
            F.col("quantity").cast("decimal(38,0)")
        ).alias("committed_return_quantity")
    )
    .join(
        published_order_lines,
        on=["order_id", "line_id"],
        how="left"
    )
)

invalid_return_totals = published_return_totals.filter(
    F.col("ordered_quantity").isNull()
    | F.col("committed_return_quantity").isNull()
    | (
        F.col("committed_return_quantity")
        > F.col("ordered_quantity")
    )
)

if invalid_return_totals.limit(1).count() > 0:
    display(invalid_return_totals)
    raise ValueError("Published return quantities exceed order quantities.")

# Gold must preserve the return IDs, including their multiplicity.
silver_return_ids = published_returns.select("return_id")
gold_return_ids = published_return_details.select("return_id")

if (
    silver_return_ids.exceptAll(gold_return_ids).limit(1).count() > 0
    or gold_return_ids.exceptAll(silver_return_ids).limit(1).count() > 0
):
    raise ValueError("Gold does not preserve all published returns.")

print("Published returns checks passed.")

In [ ]:
published_refunds = spark.table(
    f"{catalog}.silver.refunds_current"
)

published_refund_summary = spark.table(
    f"{catalog}.gold.payment_refund_summary"
)

duplicate_refunds = (
    published_refunds
    .groupBy("refund_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_refunds.limit(1).count() > 0:
    raise ValueError("Duplicate refund IDs in Silver.")

# Independently aggregate the published Silver refunds.
expected_refund_totals = (
    published_refunds
    .groupBy("payment_id", "currency")
    .agg(
        F.count("*").alias("refund_count"),

        F.sum(
            F.when(F.col("status") == "pending", F.col("amount"))
            .otherwise(F.lit(0))
        ).alias("pending_refund_amount"),

        F.sum(
            F.when(F.col("status") == "succeeded", F.col("amount"))
            .otherwise(F.lit(0))
        ).alias("succeeded_refund_amount"),

        F.sum(
            F.when(F.col("status") == "failed", F.col("amount"))
            .otherwise(F.lit(0))
        ).alias("failed_refund_amount")
    )
)

# Exclude Gold rows for payments that have no refunds.
actual_refund_totals = (
    published_refund_summary
    .filter(F.col("refund_count") > 0)
    .select(*expected_refund_totals.columns)
)

if (
    expected_refund_totals.exceptAll(
        actual_refund_totals
    ).limit(1).count() > 0

    or actual_refund_totals.exceptAll(
        expected_refund_totals
    ).limit(1).count() > 0
):
    raise ValueError(
        "Gold refund totals do not match published Silver refunds."
    )

# Check the derived amounts, including nulls.
invalid_refund_metrics = published_refund_summary.filter(
    ~F.col("net_captured_amount").eqNullSafe(
        F.col("captured_amount") - F.col("succeeded_refund_amount")
    )
    | ~F.col("remaining_refund_capacity").eqNullSafe(
        F.col("captured_amount")
        - F.col("succeeded_refund_amount")
        - F.col("pending_refund_amount")
    )
    | F.col("net_captured_amount").isNull()
    | F.col("remaining_refund_capacity").isNull()
    | (F.col("net_captured_amount") < 0)
    | (F.col("remaining_refund_capacity") < 0)
)

if invalid_refund_metrics.limit(1).count() > 0:
    display(invalid_refund_metrics)
    raise ValueError("Gold refund metrics are inconsistent.")

print("Published refund reconciliation checks passed.")

In [ ]:
print("All final pipeline checks passed.")